# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramithnayak8/ML_pipeline/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content page (`content_id`), aggregated over a trailing 90-day window ending at
data export time. There is no calendar `report_date` column in this starter slice (unlike the
warehouse's daily fact table) -- every field is "as of now," not "as of this day." Inside that
90-day window sit two back-to-back 30-day sub-windows (`*_last_30d` = the most recent 30 days,
`*_prev_30d` = the 30 days before that), used only to compute the trend label -- they are a
SUBSET of the 90-day window, not extra history.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("rows, cols:", df.shape)
print("duplicate content_id rows:", df["content_id"].duplicated().sum(), "-> grain holds if 0")
print("content_age_days minimum:", df["content_age_days"].min(),
      "(every row already >= 90 days old -- backs the 90-day window claim)")
print("impressions_90d minimum:", df["impressions_90d"].min(),
      "(every row has at least 1 impression)")


rows, cols: (30000, 44)
duplicate content_id rows: 0 -> grain holds if 0
content_age_days minimum: 90 (every row already >= 90 days old -- backs the 90-day window claim)
impressions_90d minimum: 1 (every row has at least 1 impression)


## 2. Fields: feature / label / context / excluded

**Feature** (knowable before the moment we would predict; 18 numeric + 8 categorical raw fields
-> 52 engineered columns after one-hot expansion): `search_volume`, `competition`, `cpc`,
`word_count`, `char_count`, `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`,
`log_ai_sessions_90d`, `days_with_impressions`, `days_with_sessions`, `content_age_days`,
`days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`,
`ai_traffic_pct`; plus `competition_level`, `content_type`, `main_intent`, `age_tier`,
`freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`.

**Label / proxy:** `trend_direction` (bucketed) and `trend_pct` (the raw percentage it is
bucketed from) -- `is_declining_label = (trend_direction == "down")`. Never features (notebook
02 demonstrates why).

**Context** (grouping/joins only, never features): `content_id` (one page), `client_id`
(32 clients -- the client-holdout split key).

**Excluded, with why:**
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`,
  `clicks_prev_30d`, `sessions_prev_30d` -- these six columns are the exact arithmetic
  `trend_pct` is built from (`(last30 - prev30) / prev30`). Using them as features would hand
  the model the label's own ingredients. Verified below: they sit inside the 90-day feature
  window (their sum never exceeds `impressions_90d`), which is exactly what makes them
  dangerous rather than merely redundant.
- `provider_used`, `model_used` -- metadata about which LLM generated the article. The data
  dictionary flags these explicitly as "not a model feature": they describe how content was
  produced, not how it performed, so mixing them in risks a spurious proxy for the generation
  pipeline instead of a real search/engagement signal.
- `content_id`, `client_id` -- pseudonyms; carry no signal, context-only (see above).

In [2]:
import json

res = json.load(open("../../outputs/model_results.json"))
features = set(res["model_numeric_features"]) | set(res["model_categorical_features"])

buckets = {
    "label-source": {"trend_direction", "trend_pct"},
    "window-overlap (label inputs)": {"impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                                       "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"},
    "excluded metadata": {"provider_used", "model_used"},
    "context ids": {"content_id", "client_id"},
}

for name, cols in buckets.items():
    leaked = cols & features
    print(f"{name}: leaked into model features? {sorted(leaked) if leaked else 'NONE (good)'}")

print()
print(f"Actual features used: {len(features)} raw fields -> {res['feature_count']} engineered "
      "columns (one-hot expands the categoricals)")


label-source: leaked into model features? NONE (good)
window-overlap (label inputs): leaked into model features? NONE (good)
excluded metadata: leaked into model features? NONE (good)
context ids: leaked into model features? NONE (good)

Actual features used: 26 raw fields -> 52 engineered columns (one-hot expands the categoricals)


## 3. Verify it with queries (grain, counts, missing values, windows)

Four claims from above, each checked against the actual file:
1. Grain: zero duplicate `content_id` rows.
2. Windows: every row's `content_age_days` is at least 90, and the two 30-day sub-windows never
   sum past the 90-day total -- confirming they are a subset, not extra history.
3. Missingness follows `content_type`, not chance: `feedly article` is missing `search_volume`
   for 100% of its rows (it never had a keyword to look up), while `keyword article` is missing
   it for only 1.4% -- a blind `fillna(0)` on `search_volume` would silently tell the model
   this is a feedly article.
4. `avg_position == 0` is the "no position data" code, not literal position zero -- confirmed
   below as a real, non-trivial slice of rows that needs explicit handling.

In [3]:
print("1) duplicate content_id rows:", df["content_id"].duplicated().sum())

print("\n2) content_age_days min:", df["content_age_days"].min())
sub = df[["impressions_90d", "impressions_last_30d", "impressions_prev_30d"]]
over = (sub["impressions_last_30d"] + sub["impressions_prev_30d"] > sub["impressions_90d"]).sum()
print("   rows where last30 + prev30 > 90d total:", over, "(0 confirms subset, not extra history)")

print("\n3) missing search_volume by content_type:")
print(df.groupby("content_type")["search_volume"].apply(lambda s: f"{s.isna().mean():.1%}"))
print("   missing word_count by content_type:")
print(df.groupby("content_type")["word_count"].apply(lambda s: f"{s.isna().mean():.1%}"))

print("\n4) avg_position == 0 rows (no data, not rank zero):", (df["avg_position"] == 0).sum())


1) duplicate content_id rows: 0

2) content_age_days min: 90
   rows where last30 + prev30 > 90d total: 0 (0 confirms subset, not extra history)

3) missing search_volume by content_type:
content_type
comparison article      0.0%
feedly article        100.0%
keyword article         1.4%
Name: search_volume, dtype: str
   missing word_count by content_type:
content_type
comparison article     0.0%
feedly article         0.0%
keyword article       28.3%
Name: word_count, dtype: str

4) avg_position == 0 rows (no data, not rank zero): 1205


## 4. Data limits

What this starter slice can never tell me:
- **No real calendar history.** It is one snapshot with two nested 30-day sub-windows, not a
  daily time series. A genuine "features from the past predict an outcome in the future" label
  cannot be built from this file alone -- for that I need the warehouse's
  `fact_content_daily_performance` (daily grain, roughly 17 months) in later weeks.
- **No causal claim is possible.** Nothing here shows a refresh causes recovery -- only an
  experiment could show that.
- **Cannot separate decline from consolidation or seasonality.** There is no sibling-page or
  calendar-history data in this slice to check whether a declining page's traffic actually moved
  to a related page, or just follows a seasonal dip
  (`docs/ml-intern-dataset-and-lane-guide.md`, section 7) -- I can flag decline, not explain it.
- **Missingness is systematic**, not random -- confirmed above by `content_type`. Needs
  has_-flags if used, never a blind `fillna(0)`.
- **Client sizes are wildly unbalanced** -- confirmed below. Small clients contribute almost no
  signal to a client-holdout split and make any single-client read fragile.
- **`avg_position == 0` is a missing-data code**, not a real position -- must be excluded or
  flagged, never averaged in as-is.

In [4]:
client_counts = df.groupby("client_id").size()
print(client_counts.describe()[["min", "25%", "50%", "75%", "max"]])
print(f"\nclients with fewer than 100 pages: {(client_counts < 100).sum()} of {len(client_counts)}")


min       3.00
25%     110.25
50%     567.00
75%    1058.75
max    7008.00
dtype: float64

clients with fewer than 100 pages: 8 of 32


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.